<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/Binding%20Circuit%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

In [2]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [3]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/2L_1H_Entity_Binding"
FILENAME = "2L_1H_Attn_Only.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


2L_1H_Attn_Only.pth:   0%|          | 0.00/2.63M [00:00<?, ?B/s]

In [4]:
### Model


E = 100 # num entities
A = 100 # num attributes
T = 10 # num types/relations
SEP = E+A+T # as seperator between relations
Q = E+A+T+1 # question token
PAD = E+A+T+2
D_VOCAB = E+A+T+3
IGNORE_INDEX = -100

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,
    d_model=256,
    d_head=256,
    n_ctx=64,
    d_vocab=D_VOCAB,
    act_fn="gelu",
    attn_only=True,
    #normalization_type="LN",
    use_attn_result=True
)
model = HookedTransformer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

print("Model loaded successfully.")

Moving model to device:  cpu
Model loaded successfully.


In [5]:
id_mapping_df = pd.read_csv('id_mapping.csv')
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [6]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

In [ ]:
### Some very basic checks: What happens if we pertube the input sequence

In [7]:
id_to_entity[210] = ","
id_to_entity[211] = "?"
example, label = test_dataset[0]

In [8]:
print("Example tokens and their mapped entities:")
for token_id in example:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example tokens and their mapped entities:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [ ]:
## considering the incorrect token to be the 10th most probable prediction
## take a batch of examples with this heuristic

In [9]:
try:
    import google.colab # type: ignore
    IN_COLAB = True
except:
    IN_COLAB = False

import os, sys
chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"

if IN_COLAB:
    # Install packages
    %pip install transformer_lens
    %pip install einops
    %pip install jaxtyping
    %pip install git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

    # Code to download the necessary files (e.g. solutions, test funcs)
    if not os.path.exists(f"/content/{chapter}"):
        !wget https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/main.zip
        !unzip /content/main.zip 'ARENA_3.0-main/chapter1_transformer_interp/exercises/*'
        sys.path.append(f"/content/{repo}-main/{chapter}/exercises")
        os.remove("/content/main.zip")
        os.rename(f"{repo}-main/{chapter}", chapter)
        os.rmdir(f"{repo}-main")
        os.chdir(f"{chapter}/exercises")
else:
    chapter_dir = r"./" if chapter in os.listdir() else os.getcwd().split(chapter)[0]
    sys.path.append(chapter_dir + f"{chapter}/exercises")

  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-v0l_71oq
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-v0l_71oq
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for circuitsvis: filename=circuitsvis-0.0.0-py3-none-any.whl size=6172337 sha256=a4f68a36d13a4fbe2ed1d8bde97d80050bd39c247f48ad5a9f309c1c97601ffd
  Stored in directory: /tmp/pip-ephem-wheel-cache-out0axru/wheels/00/ce/19/651aed367fa8cefad943dece40a2248cef6588697047472ef1
Successfully built circuitsvis
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 8.7.0
    Uninstalling importlib_metadata-8.7.0:
      Successfully uninstalled importlib_metadata-8.7.0
--2025-

## Direct Logit Attribution/Logit Lens

In [10]:
from rich.table import Table, Column, box
from rich import print as rprint
from jaxtyping import Float, Int, Bool
from typing import List, Optional, Callable, Tuple, Dict, Literal, Set, Union
from torch import Tensor
from transformer_lens import ActivationCache
import einops
from rich import print as rprint
from plotly_utils import imshow, line, scatter, bar
from pathlib import Path
from IPython.display import display, HTML
from transformer_lens import utils
import circuitsvis as cv


In [137]:
import random
random.seed(42)
same_seq_subset = [ex for ex in test_dataset if len(ex[0]) == 23]

start, end = 0, len(same_seq_subset)
count = 10
random_ints = [random.randint(start, end) for _ in range(count)]

dla_dataset = [same_seq_subset[idx] for idx in random_ints]


In [138]:
text_examples = []
text_labels = []
text_incorrect_answers = []

incorrect_tokens = []
label_tokens = []
example_tokens = []

top1_probs = []
top10_probs = []

logit_diffs = []
acc = 0

for ex in dla_dataset:
  # get model prediction
  input, label = ex
  with torch.no_grad():
    logits = model(input)

  probs = logits[0, -1, :].softmax(dim=-1)
  top_values, top_indices = probs.topk(10)
  incorrect_token = top_indices.tolist()[-1]

  text_incorrect_answers.append(id_to_entity[incorrect_token])
  incorrect_tokens.append(incorrect_token)

  text_examples.append(" ".join([id_to_entity[tok] for tok in ex[0].tolist()]))
  example_tokens.append(ex)

  text_labels.append(id_to_entity[label.item()])
  label_tokens.append(label.item())

  top1_probs.append(top_values.tolist()[0])
  top10_probs.append(top_values.tolist()[-1])

  correct_token = top_indices.tolist()[0]
  if correct_token == label.item():
    acc += 1

  logit_diffs.append(logits[0, -1, label.item()] - logits[0, -1, incorrect_token])

In [120]:
cols = [
    "Prompt",
    Column("Correct", style="rgb(0,200,0) bold"),
    Column("Incorrect", style="rgb(255,0,0) bold"),
    Column("Logit Difference", style="bold")
]
table = Table(*cols, title="Logit differences", show_lines=True)

for prompt, answer, incorrect_answer, logit_diff in zip(text_examples, text_labels, text_incorrect_answers, logit_diffs):
    table.add_row(prompt, repr(answer), repr(incorrect_answer), f"{logit_diff.item():.3f}")

rprint(table)

                                                 Logit differences                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Prompt                                                       ┃ Correct        ┃ Incorrect    ┃ Logit Difference ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Cassandra left Daejeon , Leonard works in Zhengzhou , Robert │ 'Incheon'      │ 'Chicago'    │ 16.935           │
│ moved to Wuhan , Christopher lives in Incheon , Keith loves  │                │              │                  │
│ Dallas , lives in Christopher ?                              │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ William was born in Guangzhou , Matthew works in Pune ,      │ 'Guangzhou'    │ 'Washington' │ 15.382           │
│ Antonio studied in Chicago , Leonard moved to Mumbai ,       │                │              │                  │
│ Jessica visited Shenzhen , was born in William ?             │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ Matthew works in Wuhan , Allison lives in Chicago , Jeremy   │ 'Philadelphia' │ 'Brasilia'   │ 12.843           │
│ visited Philadelphia , Sharon was born in Cairo , Brenda     │                │              │                  │
│ moved to Karachi , visited Jeremy ?                          │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ Diana lives in Melbourne , Jeremy travels to Manila ,        │ 'Berlin'       │ 'Istanbul'   │ 18.713           │
│ Christine visited Karachi , Bridget studied in Berlin ,      │                │              │                  │
│ Wendy was born in Baghdad , studied in Bridget ?             │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ Angela visited Osaka , Lisa lives in Kinshasa , Deborah left │ 'Osaka'        │ 'Jinan'      │ 14.746           │
│ Mumbai , Melanie studied in Qingdao , Anthony travels to     │                │              │                  │
│ Sendai , visited Angela ?                                    │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ Nicholas works in Zhengzhou , Matthew lives in Daegu , Tasha │ 'Zhengzhou'    │ 'Dhaka'      │ 16.669           │
│ loves Santiago , Teresa studied in Baghdad , Patricia left   │                │              │                  │
│ Philadelphia , works in Nicholas ?                           │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ George studied in Guangzhou , Christopher married in New     │ 'Shenzhen'     │ 'Melbourne'  │ 15.813           │
│ York , Kimberly travels to Incheon , Janet visited Shenzhen  │                │              │                  │
│ , Robert was born in Kano , visited Janet ?                  │                │              │                  │
├──────────────────────────────────────────────────────────────┼────────────────┼──────────────┼──────────────────┤
│ Michelle studied in Delhi , Michael loves Shenzhen , Tasha   │ 'Ulsan'        │ 'Delhi'      │ 17.011           │
│ left Yokohama , Keith travels to Dallas , Jeffrey moved to   │                │              │                  │
│ Ulsan , moved to Jeffrey ?                            

In [ ]:
## logit lens

In [121]:
def residual_stack_to_logit_diff(
    residual_stack: Float[Tensor, "... batch d_model"], # contains residual stream values for the final sequence position
    cache: ActivationCache,
    logit_diff_directions: Float[Tensor, "batch d_model"],
) -> Float[Tensor, "..."]:
    '''
    Gets the avg logit difference between the correct and incorrect answer for a given
    stack of components in the residual stream.
    '''
    print(residual_stack.shape)
    ln_residual_stack = cache.apply_ln_to_stack(residual_stack, layer=-1, pos_slice=-1)
    average_logit_diff = einops.einsum(ln_residual_stack, logit_diff_directions, "... batch d_model, batch d_model ->...") / residual_stack.shape[0]
    return average_logit_diff

In [139]:
clean_dataset = torch.stack([ex[0] for ex in dla_dataset], dim=0)
clean_dataset.shape

torch.Size([10, 23])

In [140]:
# this is equivalent to indexing the unembedding matrix and getting the column corresponding to the given index/token
answer_token_directions = model.tokens_to_residual_directions(torch.tensor(label_tokens))
incorrect_token_directions = model.tokens_to_residual_directions(torch.tensor(incorrect_tokens))
logit_diff_directions = answer_token_directions - incorrect_token_directions


In [141]:
# verification of the method
(answer_token_directions[0] == model.W_U[:, label_tokens[0]]).sum() == 256

tensor(True)

In [18]:
def make_WV_identity(layer):
  model.blocks[layer].attn.W_V.data.fill_(1.0)

def make_WO_identity(layer):
  model.blocks[layer].attn.W_O.data.fill_(1.0)

In [19]:
original_logits, cache = model.run_with_cache(dla_dataset)

In [20]:
accumulated_residual, labels = cache.accumulated_resid(layer=-1, pos_slice=-1, return_labels=True)
# accumulated_residual has shape (component, batch, d_model)
# 12 blocks, so 12 attn, 12 mlp and one input layer
print("acc", accumulated_residual.shape)
logit_lens_logit_diffs: Float[Tensor, "component"] = residual_stack_to_logit_diff(accumulated_residual, cache, logit_diff_directions)

line(
    logit_lens_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Accumulated Residual Stream",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800
)

acc torch.Size([3, 10, 256])
torch.Size([3, 10, 256])


In [134]:
per_layer_residual, labels = cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)
per_layer_logit_diffs = residual_stack_to_logit_diff(per_layer_residual, cache, logit_diff_directions)

line(
    per_layer_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Each Layer",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800
)

torch.Size([4, 10, 256])


In [142]:
per_head_residual, labels = cache.stack_head_results(layer=-1, pos_slice=-1, return_labels=True)
per_head_residual = einops.rearrange(
    per_head_residual,
    "(layer head) ... -> layer head ...",
    layer=model.cfg.n_layers
)
per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, cache, logit_diff_directions)

imshow(
    per_head_logit_diffs,
    labels={"x":"Head", "y":"Layer"},
    title="Logit Difference From Each Head",
    width=600
)

torch.Size([2, 1, 10, 256])


## Attention Analysis

In [22]:
def topk_of_Nd_tensor(tensor: Float[Tensor, "rows cols"], k: int):
    '''
    Helper function: does same as tensor.topk(k).indices, but works over 2D tensors.
    Returns a list of indices, i.e. shape [k, tensor.ndim].

    Example: if tensor is 2D array of values for each head in each layer, this will
    return a list of heads.
    '''
    i = torch.topk(tensor.flatten(), k).indices
    return np.array(np.unravel_index(utils.to_numpy(i), tensor.shape)).T.tolist()


k = 2

for head_type in ["Positive", "Negative"]:

    # Get the heads with largest (or smallest) contribution to the logit difference
    top_heads = topk_of_Nd_tensor(per_head_logit_diffs * (1 if head_type=="Positive" else -1), k)

    # Get all their attention patterns
    attn_patterns_for_important_heads: Float[Tensor, "head q k"] = torch.stack([
        cache["pattern", layer][:, head].mean(0)
         for layer, head in top_heads
    ])

    # Display results
    display(HTML(f"<h2>Top {k} {head_type} Logit Attribution Heads</h2>"))
    display(cv.attention.attention_patterns(
        attention = attn_patterns_for_important_heads,
        tokens = [id_to_entity[tok] for tok in dla_dataset[6].tolist()],
        attention_head_names = [f"{layer}.{head}" for layer, head in top_heads],
    ))

## Analyzing Corruption Techniques

In [23]:
clean = dla_dataset[4]
clean, " ".join([id_to_entity[tok] for tok in clean.tolist()])

(tensor([ 62, 206, 188, 210,  18, 200, 122, 210,  54, 209, 104, 210,  28, 204,
         165, 210,  12, 203, 191, 210, 206,  62, 211]),
 'Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , visited Angela ?')

In [24]:
example_tokens[0], text_examples[0]

((tensor([ 29, 209, 195, 210,  73, 202, 166, 210,  93, 208, 136, 210,  88, 200,
          198, 210,  43, 207, 152, 210, 200,  88, 211]),
  tensor(198)),
 'Cassandra left Daejeon , Leonard works in Zhengzhou , Robert moved to Wuhan , Christopher lives in Incheon , Keith loves Dallas , lives in Christopher ?')

In [25]:
corrupt_entity = 29 # Cassandra
corrupt_relation = 202 # works in
ic_relation = 200 # lives in
ic_entity = 18 # Lisa
ic_entity2 = 28 # Melanie

In [26]:
## clean
print(" ".join([id_to_entity[tok] for tok in clean.tolist()]))
## both entity and relation are not in context
corrupt_no_entity_no_rel = clean.tolist()[:-3] + [corrupt_relation, corrupt_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_entity_no_rel]))
## entity is there but relation not in context
corrupt_no_rel = clean.tolist()[:-3] + [corrupt_relation, ic_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_rel]))
## relation is there but entity not in context
corrupt_no_entity = clean.tolist()[:-3] + [ic_relation, corrupt_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_entity]))
## both entity and relaion are in context, but not binded
corrupt_both_in_context = clean.tolist()[:-3] + [203, ic_entity2, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_both_in_context]))

Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , visited Angela ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , works in Cassandra ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , works in Lisa ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , lives in Cassandra ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , travels to Melanie ?


In [27]:
def print_stats(logits):
  probs = logits[0, -1, :].softmax(dim=-1)*100
  top_values, top_indices = probs.topk(10)
  top_values, top_indices = top_values.tolist(), top_indices.tolist()
  for idx in range(10):
    print(f"{top_indices[idx]}, {id_to_entity[top_indices[idx]]}, {top_values[idx]}%")

In [28]:
clean_logits = model(clean)
corrupt1_logits = model(torch.tensor(corrupt_no_entity_no_rel))
corrupt2_logits = model(torch.tensor(corrupt_no_rel))
corrupt3_logits = model(torch.tensor(corrupt_no_entity))
corrupt4_logits = model(torch.tensor(corrupt_both_in_context))

In [29]:
print("Clean")
print_stats(clean_logits)
print("=================================================")
print("No entity no relation in context")
print_stats(corrupt1_logits)
print("==================================================")
print("No relation in context")
print_stats(corrupt2_logits)
print("==================================================")
print("No entity in context")
print_stats(corrupt3_logits)
print("==================================================")
print("No entity entity_type binding in context")
print_stats(corrupt4_logits)

Clean
188, Osaka, 94.62728118896484%
122, Kinshasa, 5.299396991729736%
104, Mumbai, 0.07225257903337479%
161, Alexandria, 8.464651909889653e-05%
190, Fukuoka, 7.243245636345819e-05%
191, Sendai, 6.574164581252262e-05%
156, Atlanta, 4.808239464182407e-05%
169, Cape Town, 3.982225825893693e-05%
100, Tokyo, 3.936935536330566e-05%
168, Jinan, 3.732922050403431e-05%
No entity no relation in context
165, Qingdao, 99.99430084228516%
104, Mumbai, 0.005540973506867886%
188, Osaka, 3.184013985446654e-05%
122, Kinshasa, 2.068803041765932e-05%
111, Sao Paulo, 1.3882109669793863e-05%
164, Shenyang, 9.33366027311422e-06%
130, Hyderabad, 7.4098452387261204e-06%
100, Tokyo, 6.960201517358655e-06%
125, Paris, 5.688731107511558e-06%
124, Bangalore, 5.012269866710994e-06%
No relation in context
165, Qingdao, 99.99366760253906%
104, Mumbai, 0.006182094570249319%
188, Osaka, 2.9884848117944784e-05%
122, Kinshasa, 1.4959457075747196e-05%
111, Sao Paulo, 1.2648165466089267e-05%
164, Shenyang, 9.1156853159191

## Activation Patching

In [110]:
def create_corrupt_example(clean_example):
  """
    Corruption process:
      1. Splits the input into parts using token `210` as a separator.
      2. Identifies the query part (last part in the split).
      3. Finds all context parts except the one that exactly matches the query's
        entity–relation pair (to avoid trivial self-copy).
      4. Randomly selects one part to supply a new entity and another to supply
        a new relation.
      5. Replaces the query's original relation and entity with the randomly
        selected ones.
      6. Returns the corrupted token sequence as a new tensor.

  """
  parts = " ".join(str(tok) for tok in clean_example.tolist()).split("210")

  query_part = parts[-1]
  all_idx = list(range(len(parts)-1))
  for idx, part in enumerate(parts[:-1]):
    if query_part.strip()[:-4].split()[::-1] == part.split()[:-1]:
      all_idx.remove(idx)
      break

  entity_idx = random.choice(all_idx)
  all_idx.remove(entity_idx)
  relation_idx = random.choice(all_idx)
  corrupt_entity_part = parts[entity_idx].strip()
  corrupt_relation_part = parts[relation_idx].strip()
  corrupt_relation = int(corrupt_relation_part.split()[1])
  corrupt_entity = int(corrupt_entity_part.split()[0])
  corrupt_example = clean_example.tolist()[:-3] + [corrupt_relation, corrupt_entity, 211]

  return torch.tensor(corrupt_example)



In [143]:
corrupt_dataset = []
for clean_example in clean_dataset:
  corrupt_dataset.append(create_corrupt_example(clean_example))

corrupt_dataset = torch.stack(corrupt_dataset)

In [144]:
def render_clean_and_corrupt(clean_dataset, corrupt_dataset):
  for idx in range(len(clean_dataset)):
    print(f"Clean: {' '.join([id_to_entity[tok] for tok in clean_dataset.tolist()[idx]])}")
    print(f"Corrupt: {' '.join([id_to_entity[tok] for tok in corrupt_dataset.tolist()[idx]])}")
    print("=============================================================================")

In [145]:
render_clean_and_corrupt(clean_dataset, corrupt_dataset)

Clean: Cassandra left Daejeon , Leonard works in Zhengzhou , Robert moved to Wuhan , Christopher lives in Incheon , Keith loves Dallas , lives in Christopher ?
Corrupt: Cassandra left Daejeon , Leonard works in Zhengzhou , Robert moved to Wuhan , Christopher lives in Incheon , Keith loves Dallas , loves Cassandra ?
Clean: William was born in Guangzhou , Matthew works in Pune , Antonio studied in Chicago , Leonard moved to Mumbai , Jessica visited Shenzhen , was born in William ?
Corrupt: William was born in Guangzhou , Matthew works in Pune , Antonio studied in Chicago , Leonard moved to Mumbai , Jessica visited Shenzhen , works in Jessica ?
Clean: Matthew works in Wuhan , Allison lives in Chicago , Jeremy visited Philadelphia , Sharon was born in Cairo , Brenda moved to Karachi , visited Jeremy ?
Corrupt: Matthew works in Wuhan , Allison lives in Chicago , Jeremy visited Philadelphia , Sharon was born in Cairo , Brenda moved to Karachi , lives in Matthew ?
Clean: Diana lives in Melbou

In [151]:
## In IOI we have a clear notion of a correct answer and an incorrect answer given an input.
## Meaning, if we flip/perturb some part of input the output flips in a deterministic manner.
## That is not the case here. Our perturbation is not principled as in IOI and is not guaranteed
## to flip the output.

## current way of getting the incorrect answers, taking the model's prediction on the corrupt dataset

logits = model(corrupt_dataset)
incorrect_answers = logits[:, -1].argmax(dim=-1)
answer_tokens = torch.stack([torch.tensor(label_tokens), incorrect_answers], dim=1)

## the way we calculated incorrect tokens for DLA - take the 10th most probable prediction
## on the clean dataset as the incorrect answer. Usually the probability of this token is super low.

# answer_tokens = torch.stack([torch.tensor(label_tokens), torch.tensor(incorrect_tokens)], dim=1)

In [158]:
def logits_to_ave_logit_diff(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Float[Tensor, "batch 2"]=answer_tokens,
    per_prompt: bool = False
) -> Union[Float[Tensor, ""], Float[Tensor, "batch"]]:
    '''
    Returns logit difference between the correct and incorrect answer.

    If per_prompt=True, return the array of differences rather than the average.
    '''
    diffs = []
    batch_size, seq = logits.shape[0], logits.shape[1]
    for i in range(batch_size):
      correct_idx = answer_tokens[i,0]
      incorrect_idx = answer_tokens[i,1]
      diff = logits[i, seq-1, correct_idx] - logits[i, seq-1, incorrect_idx]
      diffs.append(diff)
    if per_prompt:
      return torch.FloatTensor(diffs)
    else:
      return torch.FloatTensor(diffs).mean()

In [159]:
clean_logits, clean_cache = model.run_with_cache(clean_dataset)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupt_dataset)

clean_logit_diff = logits_to_ave_logit_diff(clean_logits, answer_tokens)
print(f"Clean logit diff: {clean_logit_diff:.4f}")

corrupted_logit_diff = logits_to_ave_logit_diff(corrupted_logits, answer_tokens)
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

Clean logit diff: 9.2110
Corrupted logit diff: -10.7668


In [160]:
def patching_metric(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Float[Tensor, "batch 2"]=answer_tokens,
    corrupted_logit_diff: float=corrupted_logit_diff,
    clean_logit_diff: float=clean_logit_diff,
) -> Float[Tensor, ""]:
    '''
    Linear function of logit diff, calibrated so that it equals 0 when performance is
    same as on corrupted input, and 1 when performance is same as on clean input.
    '''
    logit_diff = logits_to_ave_logit_diff(logits)
    patching_metric = logit_diff/ (clean_logit_diff + -1*corrupted_logit_diff) + 0.5
    patching_metric = torch.FloatTensor(patching_metric)
    return patching_metric

In [161]:
from transformer_lens import patching
## patching resid_pre
# Run the model with corrupted tokens at each position to get corrupted activations. We then replace these
# corrupted activations with activations from clean cache at each sequence position, to see which component/layer
# and which position improves the performance most.
act_patch_resid_pre = patching.get_act_patch_resid_pre(
    model = model,
    corrupted_tokens = corrupt_dataset, # corrupted input sentences, with S2 swapped with IO. Are we patching with corrupted tokens?
    clean_cache = clean_cache, # clean cache, so clean activations
    patching_metric = patching_metric
)

  0%|          | 0/46 [00:00<?, ?it/s]

In [164]:
labels = [f"{id_to_entity[tok]} {i}" for i, tok in enumerate(clean_dataset[0].tolist())]
imshow(
    act_patch_resid_pre,
    labels={"x": "Position", "y": "Layer"},
    x=labels,
    title="resid_pre Activation Patching",
    width=600
)

In [169]:
## patch attn_out
act_patch_attn_out = patching.get_act_patch_attn_out(
    model = model,
    corrupted_tokens = corrupt_dataset, # corrupted input sentences, with S2 swapped with IO. Are we patching with corrupted tokens?
    clean_cache = clean_cache, # clean cache, so clean activations
    patching_metric = patching_metric
)

  0%|          | 0/46 [00:00<?, ?it/s]

In [170]:
imshow(
    act_patch_attn_out,
    labels={"x": "Position", "y": "Layer"},
    x=labels,
    title="attn_out Activation Patching",
    width=600
)

In [171]:
## patch each head output specifically
act_patch_attn_head_out_all_pos = patching.get_act_patch_attn_head_out_all_pos(
    model,
    corrupt_dataset,
    clean_cache,
    patching_metric
)

  0%|          | 0/2 [00:00<?, ?it/s]

In [172]:
imshow(
    act_patch_attn_head_out_all_pos,
    labels={"y": "Layer", "x": "Head"},
    title="attn_head_out Activation Patching (All Pos)",
    width=600
)

In [174]:
"""
Rather than just patching on head output (like the previous one), it patches on:

Output (this is equivalent to patching the value the head writes to the residual stream)
Querys (i.e. the patching the query vectors, without changing the key or value vectors)
Keys
Values
Patterns (i.e. the attention patterns).
"""

act_patch_attn_head_all_pos_every = patching.get_act_patch_attn_head_all_pos_every(
    model,
    corrupt_dataset,
    clean_cache,
    patching_metric
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [175]:
imshow(
    act_patch_attn_head_all_pos_every,
    facet_col=0,
    facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
    title="Activation Patching Per Head (All Pos)",
    labels={"x": "Head", "y": "Layer"},
)